In [9]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import nbformat

In [10]:
def clean_eurostat_table(path, dataset_name):
    
    df = pd.read_excel(path)

    # remove empty Excel columns
    df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

    # rename country column
    df = df.rename(columns={"LEV_PERC (Labels)": "country"})

    # replace ":" values
    df = df.replace(r":.*", np.nan, regex=True)

    # convert columns to numeric
    cols = ["Strong", "Intermediate", "Poor"]
    df[cols] = df[cols].apply(pd.to_numeric, errors="coerce")

    # remove EU aggregate rows
    df = df[~df["country"].str.contains("European Union", na=False)]

    # drop rows where Poor is missing
    df = df.dropna(subset=["Poor"])

    # label dataset for faceting
    df["dataset"] = dataset_name

    return df

In [11]:
df1 = clean_eurostat_table("data/citizens_by_birth.xlsx", "Citizens by birth")
df2 = clean_eurostat_table("data/eu_migrants_2019.xlsx", "EU migrants")
df3 = clean_eurostat_table("data/non_eu_countries_2019.xlsx", "Non-EU migrants")

In [12]:
print("df1:", "Cyprus" in df1["country"].values)
print("df1:", "Malta" in df1["country"].values)

df1: True
df1: True


In [13]:
vmin = min(df1["Poor"].min(), df2["Poor"].min(), df3["Poor"].min())
vmax = max(df1["Poor"].max(), df2["Poor"].max(), df3["Poor"].max())

In [14]:
fig = go.Figure()

In [15]:
fig.update_layout(
    height=700,
    width=900,
    margin=dict(l=0, r=10, t=130, b=10),
    coloraxis=dict(
        colorscale=px.colors.sequential.Mint,
        cmin=vmin,
        cmax=vmax,
        colorbar=dict(
            title=dict(
                text="% answering 'Poor'",
                side="top"
            ),
            len=0.8,
            y=0.5,
            x=0.85
        )
    ),
    title=dict(
        text="Who feels less socially supported in the EU?<br><sup>Hint: migrants</sup>",
        x=0.5,
        xanchor="center"
    ),
    updatemenus=[
        dict(
            type="buttons",
            direction="right",
            buttons=list([
                dict(label="Citizens by birth",
                     method="update",
                     args=[{"visible": [True, False, False]}]),
                dict(label="EU migrants",
                     method="update",
                     args=[{"visible": [False, True, False]}]),
                dict(label="Non-EU migrants",
                     method="update",
                     args=[{"visible": [False, False, True]}]),
            ]),
            pad={"r": 10, "t": 10},
            showactive=True,
            x=0.5,
            xanchor="center",
            y=1.1,
            yanchor="top"
        )
    ],
    annotations=[
        dict(
            text="Source: Eurostat 2019, 'Overall perceived social support by sex, age and country of birth.'",
            x=0.01,
            y=0.01,
            xref="paper",
            yref="paper",
            showarrow=False,
            align="left",
            bgcolor="rgba(255,255,255,0.7)",  # ← transparency
            bordercolor="gray",
            borderwidth=0,
            font=dict(size=11)
        )
    ]
)

In [16]:
fig.add_trace(go.Choropleth(
    locations=df1["country"],
    locationmode="country names",
    z=df1["Poor"],
    coloraxis="coloraxis",
    visible=True,
    hovertext=df1["country"],
    hovertemplate="<b>%{hovertext}</b><br>" +
              "Strong: %{customdata[0]:.1f}%<br>" +
              "Intermediate: %{customdata[1]:.1f}%<br>" +
              "Poor: %{z:.1f}%<extra></extra>",
    customdata=df1[["Strong","Intermediate"]],
))


fig.add_trace(go.Choropleth(
    locations=df2["country"],
    locationmode="country names",
    z=df2["Poor"],
    coloraxis="coloraxis",
    visible=False,
    hovertext=df2["country"],
    hovertemplate="<b>%{hovertext}</b><br>" +
              "Strong: %{customdata[0]:.1f}%<br>" +
              "Intermediate: %{customdata[1]:.1f}%<br>" +
              "Poor: %{z:.1f}%<extra></extra>",
    customdata=df2[["Strong","Intermediate"]],
    showscale=False
))



fig.add_trace(go.Choropleth(
    locations=df3["country"],
    locationmode="country names",
    z=df3["Poor"],
    coloraxis="coloraxis",
    visible=False,
    hovertext=df3["country"],
    hovertemplate="<b>%{hovertext}</b><br>" +
              "Strong: %{customdata[0]:.1f}%<br>" +
              "Intermediate: %{customdata[1]:.1f}%<br>" +
              "Poor: %{z:.1f}%<extra></extra>",
    customdata=df3[["Strong","Intermediate"]],
    showscale=False
))

fig.update_geos(scope="europe",
    showcountries=True,
    countrycolor="black",
    countrywidth=0.8,
    lonaxis_range=[-15, 35],
    lataxis_range=[30, 70],
    domain=dict(x=[0, 0.92], y=[0, 1])
)

   